# Practice 03 — Linear Regression — SOLUTIONS

Worked answers. Try the learner notebook first!

In [ ]:
# Run me first. Imports + the practice toolkit (the grader and random task generators).
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')) if os.path.basename(os.getcwd()) != 'practice' else '.')
sys.path.insert(0, '.')
import torch
import torch.nn as nn
import practice_utils as pu
print('ready — torch', torch.__version__)

## Drill 1 — Recover a random line

The task hides a relationship `y = w·x + b + noise` with **fresh `w`, `b`, and noise each draw**. Train a model on `train_data()` so it predicts the hidden test set well.

In [ ]:
task = pu.regression_task(n_features=1)
task.describe()
Xtr, ytr = task.train_data()

In [ ]:
# ✅ reference solution
model = nn.Linear(1, 1)
opt = torch.optim.Adam(model.parameters(), lr=0.1)
lf = nn.MSELoss()
for _ in range(400):
    opt.zero_grad()
    lf(model(Xtr), ytr).backward()
    opt.step()

In [ ]:
# check (run after your code)
task.grade(model)

## Drill 2 — Multi-feature regression (mind the scales!)

Now 3 input features on **different scales**. A raw fit struggles; **standardize the features** (and ideally the target), train, and wrap so the model maps raw `x → y`.

In [ ]:
task = pu.regression_task(n_features=3)
task.describe()
Xtr, ytr = task.train_data()

In [ ]:
# ✅ reference solution
mx, sx = Xtr.mean(0, keepdim=True), Xtr.std(0, keepdim=True)
my, sy = ytr.mean(), ytr.std()
Xs, ys = (Xtr - mx) / sx, (ytr - my) / sy
lin = nn.Linear(3, 1)
opt = torch.optim.Adam(lin.parameters(), lr=0.05)
lf = nn.MSELoss()
for _ in range(400):
    opt.zero_grad()
    lf(lin(Xs), ys).backward()
    opt.step()
class Wrapped(nn.Module):
    def forward(self, X):
        return lin((X - mx) / sx) * sy + my
model = Wrapped()

In [ ]:
# check (run after your code)
task.grade(model)

## Drill 3 — Beat a tighter target

Same kind of task, but the bar sits close to the noise floor (you must actually fit the signal, not just predict the mean). Standardize, train longer / tune the lr.

In [ ]:
task = pu.regression_task(n_features=2, noise_mult=1.2)
task.describe()
Xtr, ytr = task.train_data()

In [ ]:
# ✅ reference solution
mx, sx = Xtr.mean(0, keepdim=True), Xtr.std(0, keepdim=True)
my, sy = ytr.mean(), ytr.std()
Xs, ys = (Xtr - mx) / sx, (ytr - my) / sy
lin = nn.Linear(2, 1)
opt = torch.optim.Adam(lin.parameters(), lr=0.05)
lf = nn.MSELoss()
for _ in range(600):
    opt.zero_grad()
    lf(lin(Xs), ys).backward()
    opt.step()
class Wrapped(nn.Module):
    def forward(self, X):
        return lin((X - mx) / sx) * sy + my
model = Wrapped()

In [ ]:
# check (run after your code)
task.grade(model)

✅ Next: classification. → `04_classification.ipynb`